In [1]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
INPUT_DIR = "/kaggle/input/datasets/ababeiraul/poiero"
train_df = pd.read_csv(f"{INPUT_DIR}/train_data.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test_data.csv")

In [4]:
EVASION_TO_CLARITY = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "General": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Dodging": "Ambivalent",
    "Deflection": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply",
}

EVASION_LABELS = list(EVASION_TO_CLARITY.keys())
CLARITY_LABELS = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

label2id = {lbl: i for i, lbl in enumerate(EVASION_LABELS)}
id2label = {i: lbl for i, lbl in enumerate(EVASION_LABELS)}
clarity2id = {lbl: i for i, lbl in enumerate(CLARITY_LABELS)}

In [5]:
def clean_ro(text):
    if pd.isna(text):
        return ""
    return (
        str(text)
        .replace("ţ", "ț").replace("ş", "ș")
        .replace("Ţ", "Ț").replace("Ş", "Ș")
        .strip()
    )

In [6]:
full_df = pd.concat(
    [
        train_df[["datapointID", "url", "question_order", "interview_question"]].assign(is_test=0),
        test_df[["datapointID", "url", "question_order", "interview_question"]].assign(is_test=1),
    ],
    ignore_index=True,
).sort_values(["url", "question_order", "datapointID"]).reset_index(drop=True)

is_consecutive = (full_df["url"] == full_df["url"].shift(1)) & (
    (full_df["question_order"] - full_df["question_order"].shift(1)) == 1
)
full_df["prev_question"] = np.where(
    is_consecutive, full_df["interview_question"].shift(1).apply(clean_ro), ""
)

prev_q_map = dict(zip(full_df["datapointID"], full_df["prev_question"]))
train_df["prev_question"] = train_df["datapointID"].map(prev_q_map)
test_df["prev_question"] = test_df["datapointID"].map(prev_q_map)

In [61]:
def head_tail_words(text, head_w=210, tail_w=210):
    words = clean_ro(text).split()
    if len(words) <= head_w + tail_w:
        return " ".join(words)
    return " ".join(words[:head_w]) + " [...] " + " ".join(words[-tail_w:])

def shorten_prev_q(text, max_w=45):
    words = clean_ro(text).split()
    if len(words) <= max_w:
        return " ".join(words)
    return " ".join(words[-max_w:])

def format_inputs(df):
    q_inputs, a_inputs = [], []
    for _, row in df.iterrows():
        pres = clean_ro(row["president"])
        ctx = clean_ro(row["context"])

        q_curr = head_tail_words(row["interview_question"], head_w=80, tail_w=60)
        q_prev = shorten_prev_q(row["prev_question"], max_w=35)

        if q_prev:
            q_str = f"[{pres} | {ctx}] Intrebare anterioara: {q_prev} | Intrebare curenta: {q_curr}"
        else:
            q_str = f"[{pres} | {ctx}] Intrebare: {q_curr}"

        a_str = head_tail_words(row["interview_answer"], head_w=170, tail_w=170)
        q_inputs.append(q_str)
        a_inputs.append(a_str)
    return q_inputs, a_inputs

In [62]:
train_df["q_input"], train_df["a_input"] = format_inputs(train_df)
test_df["q_input"], test_df["a_input"] = format_inputs(test_df)

y_evasion = train_df["evasion_label"].map(label2id).values
y_clarity = train_df["evasion_label"].map(EVASION_TO_CLARITY).map(clarity2id).values
train_df["label"] = y_evasion

In [63]:
qev = load_dataset("ailsntua/QEvasion", split="train").to_pandas()
RARE_CLASSES = [
    "Clarification",
    "Claims ignorance",
    "Declining to answer",
    "Partial/half-answer",
    "Explicit",
]
qev_rare = qev[qev["evasion_label"].isin(RARE_CLASSES)].groupby("evasion_label").head(250).reset_index(drop=True)


print(qev_rare.head(5))

                                               title                date  \
0  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
1  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
2  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
3  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
4  The President's News Conference in Hanoi, Vietnam  September 10, 2023   

         president                                                url  \
0  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
1  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
2  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
3  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
4  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   

   question_order                                 interview_question  \
0               1  Q. Of the Bid

In [24]:
mt_name = "Helsinki-NLP/opus-mt-en-ro"
mt_tok = AutoTokenizer.from_pretrained(mt_name)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(mt_name).cuda().eval()

mt_model.generation_config.max_length = None

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [25]:
def translate_batch(texts, batch_size=32, max_len=256):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t)[:1200] for t in texts[i : i + batch_size]]
        enc = mt_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_len).to("cuda")
        with torch.no_grad():
            gen = mt_model.generate(**enc, max_new_tokens=max_len)
        out.extend(mt_tok.batch_decode(gen, skip_special_tokens=True))
    return out

In [26]:
qev_rare["interview_question"] = translate_batch(qev_rare["interview_question"].tolist(), max_len=128)
qev_rare["interview_answer"] = translate_batch(qev_rare["interview_answer"].tolist(), max_len=256)

qev_rare["president"] = "Președinte SUA"
qev_rare["context"] = "Interviu"
qev_rare["prev_question"] = ""

qev_rare["q_input"], qev_rare["a_input"] = format_inputs(qev_rare)
qev_rare["label"] = qev_rare["evasion_label"].map(label2id)

extra_df = qev_rare[["evasion_label", "label", "q_input", "a_input"]].dropna().reset_index(drop=True)

OUT_CSV = "/kaggle/working/qevasion_rare_ro.csv"
extra_df.to_csv(OUT_CSV, index=False)

del mt_model, mt_tok
if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(extra_df["evasion_label"].value_counts())
print(extra_df.head(2))

evasion_label
Explicit               250
Declining to answer    145
Claims ignorance       119
Clarification           92
Partial/half-answer     79
Name: count, dtype: int64
         evasion_label  label  \
0             Explicit      0   
1  Partial/half-answer      3   

                                             q_input  \
0  [Președinte SUA | Interviu] Intrebare: Î. a ad...   
1  [Președinte SUA | Interviu] Intrebare: Î. Nu v...   

                                             a_input  
0  Ei bine, uite, în primul rând, eu sunt sincer ...  
1  Uite, cred că China are o problemă economică d...  


In [51]:
qev_rare["q_input"], qev_rare["a_input"] = format_inputs(qev_rare)

In [40]:
import os
import shutil
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

In [64]:
INPUT_DIR = "/kaggle/input/datasets/ababeiraul/poiero"
EXTRA_CSV = "/kaggle/working/qevasion_rare_ro.csv"
MODEL_NAME = "dumitrescustefan/bert-base-romanian-cased-v1"


In [65]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["q_input"],
        batch["a_input"],
        truncation="longest_first",
        max_length=512,
    )

In [66]:
test_ds = Dataset.from_pandas(test_df[["q_input", "a_input"]]).map(
    tokenize_fn, batched=True, remove_columns=["q_input", "a_input"]
)

Map:   0%|          | 0/296 [00:00<?, ? examples/s]

In [53]:
def probs9_to_probs3(probs9):
    p_clear = probs9[:, [0]].sum(axis=1)
    p_ambiv = probs9[:, [1, 2, 3, 4, 5]].sum(axis=1)
    p_nonrep = probs9[:, [6, 7, 8]].sum(axis=1)
    return np.stack([p_clear, p_ambiv, p_nonrep], axis=1)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs9 = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds9 = np.argmax(probs9, axis=1)
    f1_evasion = f1_score(labels, preds9, average="macro")

    true_clarity = np.array([clarity2id[EVASION_TO_CLARITY[id2label[l]]] for l in labels])
    pred_clarity = np.argmax(probs9_to_probs3(probs9), axis=1)
    f1_clarity = f1_score(true_clarity, pred_clarity, average="macro")

    return {
        "f1_evasion": f1_evasion,
        "f1_clarity": f1_clarity,
        "combined": 0.5 * (f1_evasion + f1_clarity),
    }

In [68]:
EXTRA_CSV = "/kaggle/working/qevasion_rare_ro.csv"
extra_df = (
    pd.read_csv(EXTRA_CSV)
    .groupby("evasion_label")
    .head(120)
    .reset_index(drop=True)
)

In [70]:
extra_df["a_input"] = extra_df["a_input"].apply(lambda x: head_tail_words(x, head_w=125, tail_w=125))

In [59]:
import gc
for var_name in ["model", "trainer"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
torch.cuda.empty_cache()

In [71]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_aug9 = np.zeros((len(train_df), 9))
test_aug9 = np.zeros((len(test_df), 9))

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_df, y_evasion)):
    print(f"FOLD {fold + 1} / 5")
    
    fold_train = pd.concat(
        [
            train_df.iloc[tr_idx][["q_input", "a_input", "label"]],
            extra_df[["q_input", "a_input", "label"]],
        ],
        ignore_index=True,
    ).sample(frac=1.0, random_state=42 + fold).reset_index(drop=True)

    tr_ds = Dataset.from_pandas(fold_train).map(
        tokenize_fn, batched=True, remove_columns=["q_input", "a_input"]
    )
    val_ds = Dataset.from_pandas(
        train_df.iloc[val_idx][["q_input", "a_input", "label"]].reset_index(drop=True)
    ).map(tokenize_fn, batched=True, remove_columns=["q_input", "a_input"])

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=9)
    out_dir = f"./polero_aug_fold_{fold}"

    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=3e-5,
        weight_decay=0.01,
        warmup_steps=20,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="combined",
        greater_is_better=True,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    val_logits = trainer.predict(val_ds).predictions
    oof_aug9[val_idx] = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()

    test_logits = trainer.predict(test_ds).predictions
    test_aug9 += torch.softmax(torch.tensor(test_logits), dim=-1).numpy() / 5.0

    shutil.rmtree(out_dir, ignore_errors=True)

np.save("oof_readerbench_aug9.npy", oof_aug9)
np.save("test_readerbench_aug9.npy", test_aug9)


==================== FOLD 1 / 5 (cu QEvasion RO) ====================


Map:   0%|          | 0/3152 [00:00<?, ? examples/s]

Map:   0%|          | 0/656 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

Epoch,Training Loss,Validation Loss,F1 Evasion,F1 Clarity,Combined
1,No log,2.478053,0.273163,0.580692,0.426928
2,No log,2.334798,0.365600,0.628504,0.497052
3,No log,2.340171,0.361300,0.630376,0.495838


... (50 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [72]:
s1 = f1_score(y_clarity, np.argmax(probs9_to_probs3(oof_aug9), axis=1), average="macro")
s2 = f1_score(y_evasion, np.argmax(oof_aug9, axis=1), average="macro")
print(f"sub1: {s1:.4f} | sub2: {s2:.4f} | mean: {0.5*(s1+s2):.4f}")

sub1: 0.6399 | sub2: 0.3253 | mean: 0.4826


In [73]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


oof_probs = np.load("oof_readerbench_aug9.npy")
test_probs = np.load("test_readerbench_aug9.npy")


assert np.all(oof_probs.sum(axis=1) > 0), "exista randuri cu 0 in oof"
if oof_probs[0].sum() > 1.5: oof_probs /= 5.0
if test_probs[0].sum() > 1.5: test_probs /= 5.0

def get_cv_score(probs9):
    s1 = f1_score(y_clarity, np.argmax(probs9_to_probs3(probs9), axis=1), average="macro")
    s2 = f1_score(y_evasion, np.argmax(probs9, axis=1), average="macro")
    return s1, s2, 0.5 * (s1 + s2)

s1_raw, s2_raw, best_score = get_cv_score(oof_probs)
print(f"5-Fold BRUT (fără calibrare) -> Sub1: {s1_raw:.4f} | Sub2: {s2_raw:.4f} | Mediu: {best_score:.4f}")

weights = np.ones(9)
for _ in range(3):
    for c in range(9):
        for mult in np.linspace(0.70, 1.65, 20):
            temp_w = weights.copy()
            temp_w[c] = mult
            _, _, sc = get_cv_score(oof_probs * temp_w)
            if sc > best_score:
                best_score = sc
                weights = temp_w

weights_final = 0.85 * weights + 0.15 * np.ones(9)

s1_cal, s2_cal, mean_cal = get_cv_score(oof_probs * weights_final)
print(f"5-Fold CALIBRAT              -> Sub1: {s1_cal:.4f} | Sub2: {s2_cal:.4f} | Mediu: {mean_cal:.4f}")
print("Multiplicatori aplicați:", dict(zip(EVASION_LABELS, np.round(weights_final, 3))))

# 3. Generăm submisia finală
final_test9 = test_probs * weights_final
final_test3 = probs9_to_probs3(final_test9)

sub1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_df["datapointID"],
    "answer": [CLARITY_LABELS[i] for i in np.argmax(final_test3, axis=1)],
})

sub2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": test_df["datapointID"],
    "answer": [id2label[i] for i in np.argmax(final_test9, axis=1)],
})

submission = pd.concat([sub1, sub2], ignore_index=True)
submission.to_csv("submission_polero_5fold_aug_cal.csv", index=False)


5-Fold BRUT (fără calibrare) -> Sub1: 0.6399 | Sub2: 0.3253 | Mediu: 0.4826
5-Fold CALIBRAT              -> Sub1: 0.6465 | Sub2: 0.3634 | Mediu: 0.5050
Multiplicatori aplicați: {'Explicit': np.float64(0.745), 'Implicit': np.float64(1.425), 'General': np.float64(0.915), 'Partial/half-answer': np.float64(0.83), 'Dodging': np.float64(0.745), 'Deflection': np.float64(1.552), 'Declining to answer': np.float64(0.915), 'Claims ignorance': np.float64(1.128), 'Clarification': np.float64(1.0)}
